$s^{(i)}$-i'th signal, $x^{(i)}$-i'th mix, we know that $x=As$ and we define $W=A^{-1}$ so $s^{(i)}=Wx^{(i)}$.\
We assume $s_i ∼ \mathcal{L}(0, 1)$, $f_\mathcal{L}(s) = \frac{1}{2}exp (−|s|)$. The likelihood of our unmixing matrix \
$l(W)=\log |W| +\sum_{j=1}^{d}\log g′(w^{T}_j x^{(i)})$\
$\nabla_W l(W)=(W^{T})^{-1}-sign(Wx^{(i)})x^{(i)T}$\
$W:=W+\alpha ((W^{T})^{-1}-sign(Wx^{(i)})x^{(i)T})$

Data comes from https://www.kaggle.com/datasets/adittoahosankabbo/ica-dataset . \
To run the algorithm you need to download the dataset and load it via scipy.io.wavfile.read('data/ica/...') as below

In [1]:
import scipy.io.wavfile 
import os
import numpy as np

In [2]:
_,a=scipy.io.wavfile.read('data/ica/Mix1.wav')
_,b=scipy.io.wavfile.read('data/ica/Mix2.wav')
rate,c=scipy.io.wavfile.read('data/ica/Mix3.wav')

In [3]:
data=np.column_stack((a,b,c))

In [4]:
def normalize(dat):
    return 0.99 * dat / np.max(np.abs(dat))

In [5]:
class ICA:
    def unmixer(self,X):
        M, N = X.shape
        W = np.eye(N)
    
        anneal = [0.1 , 0.1, 0.1, 0.05, 0.05, 0.05, 0.02, 0.02, 0.01 , 0.01, 0.005, 0.005, 0.002, 0.002, 0.001, 0.001]
        for lr in anneal:
            print(lr)
            rand = np.random.permutation(range(M))
            for i in rand:
                x = X[i]
                W = self._update_W(W, x, lr)
    
        return W
    def _update_W(self,W, x, learning_rate):
        def sign(z):
            return (z>0).astype('int32')-(z<0).astype('int32')
        n=W.shape[0]
        wx=W@np.reshape(x,(n,1))
        updated_W=W+learning_rate*(np.linalg.inv(W.T)-sign(wx)@np.reshape(x,(1,n)))
        
        return updated_W

In [6]:
if not os.path.isdir("output"):
    os.makedirs("output")

In [7]:
np.random.seed(0)
X = normalize(data)
print(X.shape)
ica=ICA()
W=ica.unmixer(X)
print(W)
np.savetxt('output/W.txt',W)
S=X@W.T
S = normalize(S)
for i in range(S.shape[1]):
    if os.path.exists(f'output/split_{i}'):
        os.unlink(f'output/split_{i}')
    scipy.io.wavfile.write(f'output/split_{i}.wav', rate, S[:, i])

(486144, 3)
0.1
0.1
0.1
0.05
0.05
0.05
0.02
0.02
0.01
0.01
0.005
0.005
0.002
0.002
0.001
0.001
[[ 64.01816833 -25.84758999 -28.42287082]
 [ 25.30239648   3.15745597 -23.09594018]
 [-88.63884623  20.13081688  73.83825883]]
